# Multilingual QA Workbench

Load an experiment, review translations, generate answers, and summarize ratings. Configure questions, models, review groups, and plots in the experiment JSON. See the [README](../README.md) for setup.

In [ ]:
from pathlib import Path
import os

import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from qa_workbench.config import load_experiment
from qa_workbench.workbench import Workbench

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
load_dotenv(ROOT / '.env')
CONFIG = ROOT / os.getenv('HEALTHQA_CONFIG', 'experiments/demo/experiment.json')
RUN_NAME = os.getenv('HEALTHQA_RUN', 'initial')
WORKFLOW = os.getenv('HEALTHQA_WORKFLOW', 'authoring')
assert WORKFLOW in {'authoring', 'rerun'}
EXECUTE = False  # Set True only when ready to make paid API calls.

experiment = load_experiment(CONFIG)
workbench = Workbench(experiment, RUN_NAME)
display(workbench.summary())

## Budget

Optional: fetch current OpenRouter pricing without making inference calls. This estimates the authored question/language matrix; returned-workbook reruns can have a smaller scope.

In [ ]:
FETCH_PRICING = False
if FETCH_PRICING:
    budget = workbench.budget()
    display(budget.overview, budget.recommended_budgets, budget.model_cost_drivers)

## Translation review

For authoring, generate suggestions and send `review/translations/generated/` from this run to reviewers. Return approved workbooks to the matching `completed/` folder. Returned-answer reruns use their existing approved translations.

In [ ]:
if WORKFLOW == 'authoring' and EXECUTE:
    translations = await workbench.translate()
    display(translations[['question_id', 'language_code', 'translation_suggestion']])
else:
    print('Translation generation is disabled or not needed for this workflow.')

In [ ]:
if WORKFLOW == 'authoring':
    approved = workbench.approvals()
    print(f'{len(approved)} approved translations out of {len(experiment.matrix)} eligible pairs.')
    display(approved.reindex(columns=['question_id', 'language_code', 'translated_question']))
else:
    plan = workbench.prepare_rerun()
    print(f'{len(plan.requests)} answers planned; {len(plan.pending)} held back.')
    display(plan.requests.groupby(['language_name', 'model_id']).size().rename('requests').reset_index())
    display(plan.pending)

## Answers

Generate only approved inputs. Results and review workbooks stay under this experiment’s run directory. Reruns preserve exactly the supplied question/language/model/repeat rows.

In [ ]:
if EXECUTE:
    answers = await (workbench.rerun() if WORKFLOW == 'rerun' else workbench.answer())
    print(f'{len(answers)} answers saved in {workbench.directory}')
    display(answers.reindex(columns=['question_id', 'language_code', 'candidate_name', 'answer_text']))
else:
    print('Set EXECUTE = True to generate answers.')

## Review summaries

Return rated workbooks to this run’s `review/<group>/completed/` directory. Source workbooks and completed assessments belong in separate folders.

In [ ]:
summaries = workbench.summarize()
for group, summary in summaries.items():
    print(group)
    display(summary)

In [ ]:
plots = experiment.document.get('plots', [])
if plots:
    figure, axes = plt.subplots(1, len(plots), figsize=(6 * len(plots), 5), squeeze=False)
    for spec, axis in zip(plots, axes[0]):
        summary = summaries[spec['review_group']]
        axis.set_title(spec['title'])
        if summary.empty:
            axis.text(0.5, 0.5, 'No completed reviews yet', ha='center', va='center')
            axis.axis('off')
            continue
        selected = summary.loc[
            summary['field_key'].eq(spec['field_key']) & summary['selected_option'].eq(spec['option'])
        ]
        if not selected.empty:
            selected.pivot(index='candidate_name', columns='language_code', values='share').fillna(0).plot.bar(ax=axis)
            axis.set_ylim(0, 1)
            axis.set_ylabel('Share of reviewed answers')
    plt.tight_layout()